<p style="text-align: left;">
  <img
    src="https://raw.githubusercontent.com/poma-ai/.github/main/assets/POMA_AI_Logo_Pink.svg"
    alt="POMA AI Logo"
    height="40"
    style="vertical-align: middle;"
  />
    <span style="display: inline-block;    margin: 0 18px;    font-size: 42px;    font-weight: 800;    line-height: 1;    transform: scale(1.5) translateY(-2px);    vertical-align: middle;">×</span>
  <img
    src="https://qdrant.tech/img/qdrant-logo.svg"
    alt="Qdrant Logo"
    height="40"
    style="vertical-align: middle;"
  />
</p>




---

## Overview

> - **Qdrant** is *the leading* open source *vector search engine* designed to handle high-dimensional (including multi- and sparse) vectors for performance and massive AI applications, advanced filtering, and production‑grade scalability.
> - **POMA AI** specializes in *document chunking that preserves semantic and structural context*, producing semantically coherent chunksets which retain layout, hierarchy, and contextual boundaries, making them well‑suited for ingestion into RAG workflows and scores with the Cheatsheet algorithm for deduplication and document content sorting after retrieval.
>
>Together, **POMA** prepares documents in a retrieval‑optimal form, while **Qdrant** stores and
>indexes those chunksets efficiently for downstream search and RAG pipelines.

This document describes how to connect **POMA's** chunking output to a **Qdrant** setup
and ingest the resulting chunksets into a collection with automatic embeddings.

---

## Getting Started

### Get yourself a POMA API Key

> **Note:** This notebook uses **PrimeCut** — the POMA pipeline that returns the full
> `.poma` result (chunks, chunksets, assets) for you to store in **your own** vector
> database. It is **not** a Grill setup (Grill is POMA's managed retrieval service and
> uses its own project Grill API keys, which will **not** work here).

- 1. Open https://console.poma-ai.com/

- 2. **Register** (or **Sign In** if you already have an account) with your e-mail address and a password.

- 3. In the console, open the **"Project settings"** tab and generate an API key there.

- 4. Set it as environment variable `POMA_API_KEY` — either upload a `.env` file next to the notebook, or (in Google Colab) add it in the **Secrets** panel (key icon in the left sidebar) with *Notebook access* enabled. The setup cell below picks up both automatically.


---

## Installation

We made the installation simple by just adding:

In [ ]:
%pip install -qq "poma[qdrant]"
%pip install -qq python-dotenv

Loading neccesary imports:

In [ ]:
import os, sys
from dotenv import load_dotenv

from poma import PrimeCut
from qdrant_client.http import models as qmodels
from poma.integrations.qdrant.qdrant_poma import PomaQdrant
# -------------------- Load example files if in colab --------------------
base="https://raw.githubusercontent.com/poma-ai/.github/main/notebooks/qdrant"; ("google.colab" in sys.modules) and [__import__("urllib.request").request.urlretrieve(f"{base}/{f}", f) for f in ("example.pdf","example.poma")]
# -------------------- Load envs: .env file or Colab secrets --------------------
if load_dotenv(): print("Loaded keys from .env")
elif "google.colab" in sys.modules:
    from google.colab import userdata; from contextlib import suppress
    for k in ("POMA_API_KEY","QDRANT_URL","QDRANT_API_KEY","OPENAI_API_KEY","OPENROUTER_API_KEY"):
        with suppress(Exception): os.environ.setdefault(k, userdata.get(k))
    print("Loaded keys from Colab secrets")
else: print("No .env found — using shell environment")
assert os.getenv("POMA_API_KEY"), "POMA_API_KEY missing — set it via .env, Colab secret, or shell env"
if not (os.getenv("QDRANT_URL") and os.getenv("QDRANT_API_KEY")): print("Note: no Qdrant Cloud credentials (QDRANT_URL / QDRANT_API_KEY) — only the local examples (in-memory / persisted path) can be processed")

---
## Necessary Credentials and API keys

### Set your POMA API key and Qdrant Credentials

For Qdrant Cloud, create a Free (forever) Tier cluster, and obtain a cluster endpoint and cluster API key from your dashboard.

https://cloud.qdrant.io/

Detailed instruction can be additionally found in the [Qdrant Managed Cloud docs](https://qdrant.tech/documentation/cloud/create-cluster/#create-a-cluster)

The setup cell above auto-detects where the notebook is running and loads keys from a `.env` file or (in Colab) from the **Secrets** panel. Only one key is required everywhere:

- `POMA_API_KEY` (required for all examples)

Needed only for some examples:

- `QDRANT_URL` + `QDRANT_API_KEY` (for the Qdrant Cloud examples — the in-memory and local-path examples run without them)
- `OPENAI_API_KEY` (for OpenAI embedding/inference examples)
- `OPENROUTER_API_KEY` (for OpenRouter examples)

---

## Ingesting files and process with POMA AI 

#### Setup POMA client 

In [ ]:
client = PrimeCut(os.environ["POMA_API_KEY"])

#### Process file with POMA AI

1. **Multi-Format Ingestion**  
POMA processes a wide range of file types. Simple text formats (e.g., TXT, MD) are ingested directly, while complex or visually structured files (e.g., PDFs or scanned documents) are processed through a Vision-Language Model to reliably extract the primary text. All non-textual elements such as images or layout components are preserved as structured assets.

2. **Text Preservation and Sanitization**  
The extracted main text is preserved exactly as written. We do not summarize, paraphrase, or modify the original wording. Structural elements like page numbers or headers are separated into assets rather than removed. After each processing step, automated validation checks ensure the text remains unchanged, providing extremely high fidelity to the source document.

3. **Human-Like Structural Organization**  
Using advanced language models, POMA analyzes the full document and organizes it the way a human would naturally structure it. The result is a hierarchical representation called the POMA Tree, which reflects logical sections and semantic relationships while keeping the original wording intact.

4. **Structure-Aware Chunking (Chunksets)**  
Each line initially forms a basic chunk. Based on the document’s structure, semantically related lines are grouped into Chunksets. This structure-aware approach preserves context better than traditional fixed-size chunking and improves retrieval performance. The Chunkset methodology is proprietary and patent-protected.

5. **POMA AI File Output**  
The final output is a complete `.poma` file, which is technically a structured ZIP archive with the `.poma` extension. This archive contains the full POMA representation, including the structured POMA Tree, individual Chunks, grouped Chunksets, and all associated assets.

By packaging all components into a single portable artifact, the `.poma` file ensures structural consistency, transportability, and reproducibility. The format is optimized for downstream RAG pipelines and GenAI applications while preserving the full integrity and structure of the original document.

Every intermediate step is validation-gated, ensuring textual fidelity before data progresses to the next stage.
This can take some time depending on the ingested document.

In [ ]:
chunk_data = client.ingest(
    "example.pdf",
    show_progress=True,
    download_dir="./",        # also keep a local copy of the .poma archive
    filename="example.poma",
)
print(f"Ingested: {len(chunk_data.chunks)} chunks, {len(chunk_data.chunksets)} chunksets")


`ingest(...)` submits the file and waits for the result in one call. It returns a typed
`PomaResult` (with `.chunks`, `.chunksets`, and `.images`), and — because we pass
`download_dir`/`filename` — also saves the raw `.poma` archive next to the notebook.

If you prefer to submit now and pick the result up later (e.g. for many files in
parallel), split it into the two underlying steps:

```python
job_id = client.submit("example.pdf")
chunk_data = client.collect(job_id, show_progress=True)
```


#### Or select ready *.poma (example) file instead

In [ ]:
chunk_data = "example.poma"

---

## Ingesting POMA AI files into Qdrant

Managed inference examples with optional configuration knobs.
For a full knob reference and raw-Qdrant comparison, see [Further Explanation](#qdrant-knobs-reference) at the end.


### Quick Usage Pattern (`PomaQdrant`)

`PomaQdrant` is a `QdrantClient` subclass with POMA defaults.
Normally you would define each model (dense/sparse) and its calling name.
For convenience, the PomaQdrant integration can take permament parameters for one dense and one sparse model to use during class instantiation.
They are then used throughout all operations for this collection.

1. Instantiate once with connection settings and your default `dense_model` and `sparse_model`.
2. Ingest with `upsert_poma_points(...)`.
3. Retrieve with `get_cheatsheets(...)`.

Our example below is based on:
- one dense vector field `dense` (`dense_model`) = DENSE_MODEL = "sentence-transformers/all-minilm-l6-v2"
- one sparse vector field `sparse` (`sparse_model`) = SPARSE_MODEL = "Qdrant/bm25"

Extra Qdrant knobs can be passed via `**kwargs`; when the same key exists in both places, `kwargs` overrides convenience parameters.

For full knob mapping and differences vs raw Qdrant usage, see [Further Explanation](#qdrant-knobs-reference) in the last cell.

In [ ]:
QDRANT_COLLECTION_NAME = "cloud_hybrid"
DENSE_MODEL = "sentence-transformers/all-minilm-l6-v2"
SPARSE_MODEL = "Qdrant/bm25"
DENSE_OPTIONS = {"dimensions": 384}

poma_qdrant = PomaQdrant(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    cloud_inference=True,
    timeout=120,
    collection_name=QDRANT_COLLECTION_NAME,
    dense_model=DENSE_MODEL,
    sparse_model=SPARSE_MODEL,
    dense_size=384,
    dense_options=DENSE_OPTIONS,
    auto_create_collection=True, # create collection if it doesn't exist
)

### Ingest results in the Qdrant collection

2. Ingest with `upsert_poma_points(...)` using a typed `PomaResult`, a `.poma` path, or a legacy chunk_data dict.

In [ ]:
poma_qdrant.upsert_poma_points(chunk_data)

## Retrieval:

### For each query generate structure-preserving cheatsheets

3. Retrieve with `get_cheatsheets(query=...)` for a simple default retrieval path.

With the above configuration, this call runs a hybrid retrieval using the convenience parameters we set above:
- native fusion with RRF

Then it converts Qdrant hits into structure-preserving POMA cheatsheets.

In [ ]:
cheatsheets = poma_qdrant.get_cheatsheets(
    query="Whats the positional embeddings frequency?",
    limit=10,
)

for i, cs in enumerate(cheatsheets, 1):
    print(f"\n=== Cheatsheet {i} ===")
    print(f"file_id: {cs['file_id']}")
    print(f"tag: {cs['tag']}")
    print("content:")
    print(cs["content"])


4. Change querying to `get_cheatsheets(query_obj=..., prefetch=...)` when you want direct advanced Qdrant query control.

- `query_obj`: the top-level Qdrant query object (for example `FusionQuery`, `RrfQuery`, `FormulaQuery`, `SampleQuery`, `DiscoverQuery`, `ContextQuery`).
- `prefetch`: optional list of retrieval branches (`qmodels.Prefetch(...)`) that feed into `query_obj`.
- `using`: optional vector field name for single-branch queries.
- Extra query knobs can be passed via `**kwargs`; they are forwarded to `query_points(...)` and override convenience defaults.

See [Further Explanation](#qdrant-knobs-reference) for detailed examples.

In [ ]:
query_text = "Whats the positional embeddings frequency?"

query_obj = qmodels.RrfQuery(rrf=qmodels.Rrf(k=60))
prefetch = [
    qmodels.Prefetch(
        query=qmodels.Document(
            text=query_text,
            model=DENSE_MODEL,
            options=DENSE_OPTIONS,
        ),
        using="dense",
        limit=100,
    ),
    qmodels.Prefetch(
        query=qmodels.Document(
            text=query_text,
            model=SPARSE_MODEL,
        ),
        using="sparse",
        limit=100,
    ),
]

cheatsheets = poma_qdrant.get_cheatsheets(
    query_obj=query_obj,
    prefetch=prefetch,
    collection_name=QDRANT_COLLECTION_NAME,
    limit=10,
    chunk_data=chunk_data,
)

for i, cs in enumerate(cheatsheets, 1):
    print(f"\n=== Advanced Cheatsheet {i} ===")
    print(f"file_id: {cs['file_id']}")
    print(f"tag: {cs['tag']}")
    print("content:")
    print(cs["content"])


---

## Appendix



#### More constructor examples (increasing specificity)

The examples below go from minimal defaults to fully explicit advanced configuration.

##### Current Convenience Limits

Out of the box, `upsert_poma_points(...)` builds:
- one dense vector field
- one optional sparse vector field

Use a custom pipeline (inherited raw Qdrant methods) when you need:
- multiple dense fields (for example `dense_v1` and `dense_v2`)
- advanced collection tuning (sharding/HNSW/quantization/strict mode)
- fully custom advanced query orchestration and response contracts

##### Example 1: Memory inference minimal

In [ ]:
QDRANT_COLLECTION_NAME="in_memory_fast_embed"
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DENSE_OPTIONS = None
SPARSE_MODEL = "Qdrant/bm25"

poma_qdrant = PomaQdrant(
    location=":memory:", # in-memory (RAM) inference, no persistent storage
    collection_name=QDRANT_COLLECTION_NAME,
    dense_model=DENSE_MODEL,
    sparse_model=SPARSE_MODEL,
    dense_size=384,
    auto_create_collection=True,
)


##### Example 2: Persisted path inference minimal

In [ ]:
QDRANT_COLLECTION_NAME="persistet_local_fast_embed"
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DENSE_OPTIONS = None
SPARSE_MODEL = "Qdrant/bm25"

poma_qdrant = PomaQdrant(
    path="./.qdrant_data",  # persistent local storage folder
    collection_name=QDRANT_COLLECTION_NAME,
    dense_model=DENSE_MODEL,
    sparse_model=SPARSE_MODEL,
    dense_size=384,
    auto_create_collection=True,
)


##### Example 3:  Setup the PomaQdrant client for the RAG (basic example for online inference and local FastEmbed)

In [ ]:
QDRANT_COLLECTION_NAME="online_fast_embed"
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
DENSE_OPTIONS = None
SPARSE_MODEL = "Qdrant/bm25"

poma_qdrant = PomaQdrant(
    url=os.environ["QDRANT_URL"], # cluster endpoint => online inference
    api_key=os.environ["QDRANT_API_KEY"],
    cloud_inference=False, # cloud inference = False => local FastEmbed
    collection_name=QDRANT_COLLECTION_NAME,
    dense_model=DENSE_MODEL,
    sparse_model=SPARSE_MODEL,
    dense_size=384,
    auto_create_collection=True,
)


#### Setup the PomaQdrant client for the RAG (basic example for cloud inference, defaults to OpenAI embeddings)

OPENAI_API_KEY=**required!** in DENSE_OPTIONS

##### Example 4: Cloud inference minimal (ONLY dense mode)

In [ ]:
QDRANT_COLLECTION_NAME = "cloud_dense_vectors_only"
DENSE_MODEL = "openai/text-embedding-3-large"
DENSE_OPTIONS = {"dimensions": 1536, "openai-api-key": os.environ["OPENAI_API_KEY"]}
SPARSE_MODEL = None # no sparse model set = None

poma_qdrant = PomaQdrant(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    cloud_inference=True, # cloud inference = True => no FastEmbed
    collection_name=QDRANT_COLLECTION_NAME,
    dense_model=DENSE_MODEL,
    sparse_model=SPARSE_MODEL, # no sparse model set = None
    dense_size=1536,
    dense_options=DENSE_OPTIONS,
    auto_create_collection=True,
)

##### Example 5: Cloud hybrid inference (dense + sparse explicit)

In [ ]:
QDRANT_COLLECTION_NAME = "cloud_hybrid_explicit"
DENSE_MODEL = "openai/text-embedding-3-large"
DENSE_OPTIONS = {"dimensions": 1536, "openai-api-key": os.environ["OPENAI_API_KEY"]}
SPARSE_MODEL = "Qdrant/bm25"

poma_qdrant = PomaQdrant(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    cloud_inference=True, # cloud inference = True => no FastEmbed
    collection_name=QDRANT_COLLECTION_NAME,
    dense_model=DENSE_MODEL, # set dense model explicitly
    sparse_model=SPARSE_MODEL, # set sparse model explicitly
    dense_name="dense",
    sparse_name="sparse",
    dense_size=1536,
    dense_options=DENSE_OPTIONS,
    auto_create_collection=True,
)

##### Example 6: Non-cloud fastembed (self-hosted/local endpoint)

You can also host an qdrant instance locally, to do so change the `QDRANT_URL` to the local url.
This example would except in colab, if you replace the LOCAL envs.

In [ ]:
QDRANT_COLLECTION_NAME = "local_fastembed"
LOCAL_QDRANT_URL = "http://localhost:6333"
LOCAL_QDRANT_API_KEY = "1234567890"

DENSE_MODEL = "jinaai/jina-embeddings-v2-base-en"
DENSE_OPTIONS = None
SPARSE_MODEL = "Qdrant/bm25"

poma_qdrant = PomaQdrant(
    url=os.environ["QDRANT_URL"], # replace with LOCAL_QDRANT_URL
    api_key=os.environ["QDRANT_API_KEY"], # replace with LOCAL_QDRANT_API_KEY
    cloud_inference=False, # cloud inference = False => FastEmbed
    collection_name=QDRANT_COLLECTION_NAME,
    dense_model=DENSE_MODEL,
    sparse_model=SPARSE_MODEL,
    dense_name="dense",
    sparse_name="sparse",
    dense_size=768,
    auto_create_collection=True,
)

### Advanced Retrieval

##### Example 7: Persisted storage with **external** embeddings

Additional dependencies needed in this example.

This example uses true external embedding generation (OpenAI client), then writes/query vectors directly through the current `PomaQdrant` client (inherited raw Qdrant methods).


In [ ]:
%pip install -q openai

In [ ]:
from poma.integrations.qdrant.qdrant_poma_utils import prepare_points_from_chunk_data
from qdrant_client.http import models as qmodels

from openai import OpenAI # type: ignore
from typing import Sequence

QDRANT_COLLECTION_NAME = "external_openai_dense"
DENSE_MODEL = "text-embedding-3-small"
DENSE_OPTIONS = {"dimensions": 1536}
SPARSE_MODEL = None

poma_qdrant = PomaQdrant(
    path="./.qdrant_data_2",
    cloud_inference=False,   # external embeddings, not Qdrant-managed inference
    collection_name=QDRANT_COLLECTION_NAME,
    dense_model=DENSE_MODEL,
    sparse_model=SPARSE_MODEL,
    dense_name="dense",
    sparse_name="sparse",
    dense_size=1536,
)

# Create local collection with dense vector field
if not poma_qdrant.collection_exists(QDRANT_COLLECTION_NAME):
    poma_qdrant.create_collection(
        collection_name=QDRANT_COLLECTION_NAME,
        vectors_config={
            poma_qdrant.dense_name: qmodels.VectorParams(
                size=poma_qdrant.dense_size,
                distance=qmodels.Distance.COSINE
            )
        }
    )

# generate dense vectors with external embedding model (OpenAI)
def openai_dense_embed(texts: Sequence[str], *, dimensions: int = 1536) -> list[list[float]]:
    client = OpenAI()
    res = client.embeddings.create(
        model="text-embedding-3-small",  # 1536 dims
        input=list(texts),
        dimensions=dimensions,
    )
    return [item.embedding for item in res.data]

ids, texts, payloads = prepare_points_from_chunk_data(chunk_data)
varying_vectors = openai_dense_embed(texts, dimensions=1536)

points = [
    qmodels.PointStruct(id=pid, vector={poma_qdrant.dense_name: vec}, payload=payload)
    for pid, vec, payload in zip(ids, varying_vectors, payloads, strict=True)
]
poma_qdrant.upload_points(collection_name=QDRANT_COLLECTION_NAME, points=points, wait=True)

##### Example 7.1: Querying with **external** embeddings

In [ ]:
# Dense-vector query + cheatsheet conversion
query_text = "Whats the positional embeddings frequency?"

# 1) Embed the query externally (same model/dimension you used for ingestion)
query_vector = openai_dense_embed([query_text], dimensions=1536)[0]

# 2) Query Qdrant by raw dense vector
results = poma_qdrant.query_points(
    collection_name=QDRANT_COLLECTION_NAME,
    query=query_vector, # pass dense vector directly
    using="dense",
    limit=10,
    with_payload=True,
)

# 3) Convert Qdrant hits to POMA cheatsheets
cheatsheets = poma_qdrant.get_cheatsheets(
    results=results,
)

for i, cs in enumerate(cheatsheets, 1):
    print(f"\n=== Cheatsheet {i} ===")
    print(f"file_id: {cs['file_id']}")
    print(f"tag: {cs['tag']}")
    print("content:")
    print(cs["content"])

##### Example 8: Custom Multi Vector Example: 2 Dense + 1 Sparse Fields

Use this pattern when you need more than the convenience defaults (`dense` + optional `sparse`).

This example bypasses `upsert_poma_points(...)` convenience vector building and uses raw Qdrant methods on top of `PomaQdrant`:
- explicit collection schema with 3 named vector fields
- custom point vectors per field
- fused retrieval across all branches

Notes:
- Keep using `PomaQdrant` for connection + cheatsheet conversion.
- For advanced retrieval, call `query_points(...)` directly, then convert via `get_cheatsheets(results=...)`.


In [ ]:
import os
from qdrant_client.http import models as qmodels
from poma.integrations.qdrant.qdrant_poma_utils import prepare_points_from_chunk_data

COLLECTION_MULTI = "custom_pipeline_multi_vector"

DENSE_A_MODEL = "openai/text-embedding-3-large"
DENSE_B_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_A_MODEL = "Qdrant/bm25"

DENSE_A_NAME = "dense_a"
DENSE_B_NAME = "dense_b"
SPARSE_A_NAME = "sparse_a"

DENSE_A_SIZE = 1536
DENSE_B_SIZE = 384

DENSE_A_OPTIONS = {"dimensions": DENSE_A_SIZE, "openai-api-key": os.environ["OPENAI_API_KEY"]}

poma_qdrant = PomaQdrant(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"],
    cloud_inference=True,
    collection_name=COLLECTION_MULTI,
    timeout=120, # set timeout reasonably high, due to more embedding complexity
)

# ---------- Create collection ----------
if not poma_qdrant.collection_exists(COLLECTION_MULTI):
    poma_qdrant.create_collection(
        collection_name=COLLECTION_MULTI,
        vectors_config={
            DENSE_A_NAME: qmodels.VectorParams(size=DENSE_A_SIZE, distance=qmodels.Distance.COSINE),
            DENSE_B_NAME: qmodels.VectorParams(size=DENSE_B_SIZE, distance=qmodels.Distance.COSINE),
        },
        sparse_vectors_config={
            SPARSE_A_NAME: qmodels.SparseVectorParams(),
        },
    )

# ---------- Ingest ----------
ids, texts, payloads = prepare_points_from_chunk_data(chunk_data)
points = [
    qmodels.PointStruct(
        id=pid,
        vector={
            DENSE_A_NAME: qmodels.Document(text=text, model=DENSE_A_MODEL, options=DENSE_A_OPTIONS),
            DENSE_B_NAME: qmodels.Document(text=text, model=DENSE_B_MODEL),
            SPARSE_A_NAME: qmodels.Document(text=text, model=SPARSE_A_MODEL),
        },
        payload=payload,
    )
    for pid, text, payload in zip(ids, texts, payloads, strict=True)
]

poma_qdrant.upload_points(collection_name=COLLECTION_MULTI, points=points, wait=True)

# ---------- Query ----------
query_text = "Whats the positional embeddings frequency?"
prefetch = [
    qmodels.Prefetch(
        query=qmodels.Document(text=query_text, model=DENSE_A_MODEL, options=DENSE_A_OPTIONS),
        using=DENSE_A_NAME,
        limit=100,
    ),
    qmodels.Prefetch(
        query=qmodels.Document(text=query_text, model=DENSE_B_MODEL),
        using=DENSE_B_NAME,
        limit=100,
    ),
    qmodels.Prefetch(
        query=qmodels.Document(text=query_text, model=SPARSE_A_MODEL),
        using=SPARSE_A_NAME,
        limit=100,
    ),
]

results = poma_qdrant.query_points(
    collection_name=COLLECTION_MULTI,
    query=qmodels.FusionQuery(fusion=qmodels.Fusion.RRF), #qmodels.FusionQuery(fusion=qmodels.Fusion.DBSF),
    prefetch=prefetch,
    limit=5,
    with_payload=True,
)

# ---------- Convert to cheatsheets ----------
cheatsheets = poma_qdrant.get_cheatsheets(results=results)

for i, cs in enumerate(cheatsheets, 1):
    print(f"\n=== Custom Multi-Vector Cheatsheet {i} ===")
    print(f"file_id: {cs.get('file_id')}")
    print(f"tag: {cs.get('tag')}")
    print(cs.get("content", ""))

##### Prints FastEmbed models

In [ ]:
from qdrant_client import QdrantClient
l = list(QdrantClient.list_text_models().keys()); print("FastEmbed dense models:", *[l[i:i+3] for i in range(0, len(l), 3)], sep="\n")
l = list(QdrantClient.list_sparse_models().keys()); print("FastEmbed sparse models:", *[l[i:i+3] for i in range(0, len(l), 3)], sep="\n")

---
<a id="qdrant-knobs-reference"></a>

### Further Explanation: `PomaQdrant` Knobs vs Raw Qdrant

What `PomaQdrant(...)` configures directly:
- Qdrant connection/client knobs in this wrapper: `location`, `url`, `port`, `grpc_port`, `prefer_grpc`, `https`, `api_key`, `prefix`, `timeout`, `host`, `path`, `force_disable_check_same_thread`, `grpc_options`, `auth_token_provider`, `cloud_inference`, `local_inference_batch_size`, `check_compatibility`, `pool_size`.
- Additional passthrough: `**kwargs` is forwarded to `QdrantClient(...)` for extra/forward-compatible client options.
- POMA convenience defaults: `collection_name`, `dense_model`, `sparse_model`, `dense_name`, `sparse_name`, `dense_options`, `sparse_options`, `dense_size`, `distance`, `store_chunk_details`, `auto_create_collection`.

Out-of-the-box convenience behavior:
- One dense vector field (`dense_name`, default `"dense"`).
- One optional sparse field (`sparse_name`, default `"sparse"`).
- Query convenience mode (`get_cheatsheets(query=...)`):
  - If `sparse_model` is set: hybrid prefetch + `FusionQuery(Fusion.RRF)`.
  - If `sparse_model=None`: dense-only `Document` query on `dense_name`.

`kwargs` passthrough and precedence:
- `upsert_poma_points(..., **kwargs)` forwards to `self.upsert(...)`.
- `get_cheatsheets(..., **kwargs)` forwards to `self.query_points(...)`.
- If both convenience params and `kwargs` set the same forwarded key, `kwargs` wins.

### Advanced query control with `query_obj` and `prefetch`

Use this when you want explicit Qdrant query orchestration while still returning cheatsheets.

- `query_obj` is passed as the `query` argument to `query_points(...)`.
- `prefetch` is passed unchanged to `query_points(...)`.
- You can choose fusion type (`RRF`, `DBSF`) and use query objects beyond convenience mode.

```python
from qdrant_client.http import models as qmodels

query_text = "Whats the positional embeddings frequency?"

query_obj = qmodels.FusionQuery(fusion=qmodels.Fusion.RRF)
# Alternative: qmodels.FusionQuery(fusion=qmodels.Fusion.DBSF)

prefetch = [
    qmodels.Prefetch(
        query=qmodels.Document(text=query_text, model=DENSE_MODEL, options=DENSE_OPTIONS),
        using="dense",
        limit=100,
    ),
    qmodels.Prefetch(
        query=qmodels.Document(text=query_text, model=SPARSE_MODEL),
        using="sparse",
        limit=100,
    ),
]

cheatsheets = poma_qdrant.get_cheatsheets(
    collection_name=QDRANT_COLLECTION_NAME,
    query_obj=query_obj,
    prefetch=prefetch,
    limit=5,
    with_vectors=False,
)
```

When to use raw Qdrant methods directly:
- You need workflows outside the convenience defaults (for example more than one dense/sparse field in the convenience API, or fully custom retrieval orchestration).
- Use inherited methods directly (`create_collection`, `upsert`/`upload_points`, `query_points`, etc.) or call `get_cheatsheets(query_obj=..., prefetch=...)` for advanced control.